# Installation requeriments

In [ ]:
!pip install google-cloud-bigquery
!pip install xlsxwriter

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 4.3 MB/s eta 0:00:00


In [ ]:
# Auth section
# Check the possible error for auth (like invalid session)

# Note: Auth comes from google.colab. Based on documentation of bigquery, we need the Google Bigquery CLI the create a easier solution.


# Function section


# Schema models



# Step 1: Run for basic metrics. Accept the to get more information

In [ ]:
from typing import TypedDict, Optional, List, Union, Dict, Any
from abc import ABC, abstractmethod
from enum import Enum
from google.cloud.bigquery import SchemaField as gcpSchemField

class EDATypeDescription(Enum):
    """
    Enumeración para los tipos de descripción de Análisis Exploratorio de Datos (EDA).
    Define los valores permitidos para el atributo 'type_description' en las descripciones de campo.
    """
    INDEX = "INDEX"
    NUMERICAL = "NUMERICAL"
    CATEGORICAL = "CATEGORICAL" # Corregido de 'CATEGORIAL' por convención y claridad
    TEMPORAL = "TEMPORAL"
    GENERIC = "GENERIC"

    # Opcional: Un método para obtener una lista de todos los valores del enum
    @classmethod
    def list_values(cls) -> List[str]:
        return [member.value for member in cls]
# --- 2. Clases Definidas ---

# Clase Base Abstracta para las descripciones de campos EDA
class FieldEDADescription(ABC):
  """
  Clase base abstracta para todas las descripciones de campos de EDA.
  Define los atributos comunes y un método abstracto para convertir a TypedDict.
  """
  def __init__(self, field_schema: gcpSchemField):
    self._name = field_schema.name
    self._field_type = field_schema.field_type
    self._mode = field_schema.mode

  def generate_calculation(self) -> dict:
    pass


  @abstractmethod
  def to_dict(self) -> dict:
    """Convierte la instancia de la clase a su TypedDict correspondiente."""
    pass

  def __repr__(self) -> str:
      return f"{self.__class__.__name__}(name='{self.name}', type_description='{self.type_description}')"

# Clases Concretas para cada tipo de descripción EDA
class IndexFieldDescription(FieldEDADescription):
  """Descripción EDA para campos de tipo índice."""
  _TYPE_DESCRIPTION: EDATypeDescription = EDATypeDescription.INDEX

  def __init__(self, field_schema: gcpSchemField):
    super().__init__(field_schema)

  @property
  def type_description(self) -> str:
    return self._TYPE_DESCRIPTION

  def to_dict(self) -> dict:
    pass

  def generate_calculation(self) -> dict:
    return {
        f"{self._name}_missing_values" : f"SUM(CASE WHEN {self._name} IS NULL THEN 1 ELSE 0 END)",
        f"{self._name}_unique_values" : f"COUNT(DISTINCT {self._name})",
        f"{self._name}_count_values" : f"COUNT({self._name})"
    }



class NumericalFieldDescription(FieldEDADescription):
  """Descripción EDA para campos numéricos."""
  _TYPE_DESCRIPTION: EDATypeDescription  = EDATypeDescription.NUMERICAL

  def __init__(self, field_schema: gcpSchemField):
    super().__init__(field_schema)

  @property
  def type_description(self) -> str:
    return self._TYPE_DESCRIPTION

  def to_dict(self) -> dict:
    pass

  def generate_calculation(self) -> dict:
    return {
        f"{self._name}_missing_values" : f"SUM(CASE WHEN {self._name} IS NULL THEN 1 ELSE 0 END)",
        f"{self._name}_min_value" : f"MIN({self._name})",
        f"{self._name}_max_value" : f"MAX({self._name})",
        f"{self._name}_mean_value" : f"AVG({self._name})",
        f"{self._name}_std_value" : f"STDDEV({self._name})",
        f"{self._name}_count_values" : f"COUNT({self._name})"
    }

class CategoricalFieldDescription(FieldEDADescription):
  """Descripción EDA para campos categóricos."""
  _TYPE_DESCRIPTION: EDATypeDescription = EDATypeDescription.CATEGORICAL

  def __init__(self, field_schema: gcpSchemField):
    super().__init__(field_schema)

  @property
  def type_description(self) -> str:
    return self._TYPE_DESCRIPTION

  def to_dict(self) -> dict:
    pass

  def generate_calculation(self) -> dict:
    return {
        f"{self._name}_missing_values" : f"SUM(CASE WHEN {self._name} IS NULL THEN 1 ELSE 0 END)",
        f"{self._name}_count_values" : f"COUNT({self._name})",
        f"{self._name}_unique_values" : f"COUNT(DISTINCT {self._name})",
        # f"{self._name}_most_common_value" : f"MODE({self._name})"
    }

class TemporalFieldDescription(FieldEDADescription):
  """Descripción EDA para campos temporales."""
  _TYPE_DESCRIPTION: EDATypeDescription = EDATypeDescription.TEMPORAL

  def __init__(self, field_schema: gcpSchemField):
    super().__init__(field_schema)

  @property
  def type_description(self) -> str:
    return self._TYPE_DESCRIPTION

  def to_dict(self) -> dict:
    pass

  def generate_calculation(self) -> dict:
    return {
        f"{self._name}_missing_values" : f"SUM(CASE WHEN {self._name} IS NULL THEN 1 ELSE 0 END)",
        f"{self._name}_min_value" : f"MIN({self._name})",
        f"{self._name}_max_value" : f"MAX({self._name})"
    }

class GenericFieldDescription(FieldEDADescription):
  """Descripción EDA para campos genéricos/desconocidos."""
  _TYPE_DESCRIPTION: EDATypeDescription = EDATypeDescription.GENERIC

  def __init__(self, field_schema: gcpSchemField):
    super().__init__(field_schema)

  @property
  def type_description(self) -> str:
    return self._TYPE_DESCRIPTION

  def to_dict(self) -> dict:
    pass

  def generate_calculation(self) -> dict:
    return {
        f"{self._name}_missing_values" : f"SUM(CASE WHEN {self._name} IS NULL THEN 1 ELSE 0 END)",
        f"{self._name}_count_values" : f"COUNT({self._name})"
    }

# Clase para el Esquema de un Campo
class EDASchemaField:
  """
  Representa la descripción completa de un campo dentro de un esquema de tabla,
  incluyendo sus metadatos básicos y su descripción EDA detallada.
  """
  def __init__(self, name: str, field_type: str, mode: str,
               eda_description: FieldEDADescription, description: Optional[str] = None):
    self.name = name
    self.field_type = field_type
    self.mode = mode
    self.description = description
    self.eda_description = eda_description # Una instancia de una de las clases FieldEDADescription

  def to_dict(self) -> dict:
    """Convierte la instancia a su TypedDict correspondiente."""
    pass

  def generate_calculation(self) -> dict:
    return self.eda_description.generate_calculation()

  def __repr__(self) -> str:
      return (f"SchemaField(name='{self.name}', type='{self.field_type}', "
              f"mode='{self.mode}', eda_description={self.eda_description})")


# Clase para el Esquema de una Tabla
class SchemaTable:
  """
  Representa el esquema completo de una tabla, conteniendo una lista de SchemaField.
  """
  def __init__(self, name: str, fields: List[EDASchemaField], error: dict, stats : dict, table_id: str):
    self.name = name
    self.fields = fields
    self.error = error
    self.stats = stats
    self.table_id = table_id

  def generate_query(self) -> str:
    calculations: list[dict] = []
    for field in self.fields:
      field_name = field.name
      params_to_calculate = field.generate_calculation()
      calculations.append(params_to_calculate)

    calculations = reduce(lambda a, b: {**a, **b}, calculations)

    make_query_param = lambda key, value: f"{value} as {key},"


    query_params = "\n".join([make_query_param(key, value) for key, value in calculations.items()])

    query_base = f"""
      SELECT
        {query_params}
      FROM
        {self.table_id}
    """
    return query_base




  def to_dict(self) -> dict:
    """Convierte la instancia a su TypedDict correspondiente."""
    pass

  def __repr__(self) -> str:
      return f"SchemaTable(name='{self.name}', fields_count={len(self.fields)})"



class DescriptionOperator:
  def wrapper_select_description_object(field: gcpSchemField, type_dex: EDATypeDescription) -> FieldEDADescription:
    match type_dex:
      case EDATypeDescription.INDEX:
        return IndexFieldDescription(field)
      case EDATypeDescription.NUMERICAL:
        return NumericalFieldDescription(field)
      case EDATypeDescription.CATEGORICAL:
        return CategoricalFieldDescription(field)
      case EDATypeDescription.TEMPORAL:
        return TemporalFieldDescription(field)
      case EDATypeDescription.GENERIC:
        return GenericFieldDescription(field)
      case _:
        raise ValueError(f"Tipo de descripción no válido: {type_dex}")

  def select_description_object(field: gcpSchemField) -> EDATypeDescription:
    name = field.name
    field_type = field.field_type
  # def select_description_object(name, file_type) -> EDATypeDescription:

    temporal_types = ["TIMESTAMP", "DATE", "DATETIME"]
    #TODO: Check missing types
    if field_type in temporal_types:
      return EDATypeDescription.TEMPORAL
    elif name.startswith("idu") and field_type in ["INTEGER", "STRING"]:
      return EDATypeDescription.INDEX
    elif name.startswith("num") and ( "cliente" in name.lower() or "colaborador" in name.lower() or "tienda" in name.lower() or "folio" in name.lower()) :
      return EDATypeDescription.INDEX
    elif name.startswith("num") and ( "anio" in name.lower() or "mes" in name.lower() or "semana" in name.lower() or "hora" in name.lower()) :
      return EDATypeDescription.TEMPORAL
    elif name.startswith("num") and ( "etapa" in name.lower() or "tipo" in name.lower() or "status" in name.lower() or "nivel" in name.lower() or "codigo" in name.lower() or "guia" in name.lower() or "telefono" in name.lower() or "lote" in name.lower() or "empleado" in name.lower() or "referencia" in name.lower()) :
      return EDATypeDescription.CATEGORICAL
    elif name.startswith("num") and ( "proveedor" in name.lower() or  "comprador" in name.lower() ) :
      return EDATypeDescription.INDEX
    elif name.startswith("num") and field_type in ["FLOAT"] :
      return EDATypeDescription.NUMERICAL
    elif name.startswith("nom_") and field_type in ["STRING"]:
      return EDATypeDescription.CATEGORICAL
    elif name.startswith("imp_") and field_type in ["STRING"]:
      return EDATypeDescription.CATEGORICAL
    elif name.startswith("imp_") and field_type in ["INTEGER", "FLOAT"]:
      return EDATypeDescription.NUMERICAL
    elif name.startswith("id") and ( "status" in name.lower() ) :
      return EDATypeDescription.CATEGORICAL
    elif name.startswith("id_") :
      return EDATypeDescription.CATEGORICAL
    elif name.startswith("des"):
      return EDATypeDescription.CATEGORICAL
    elif name.startswith("fec"):
      return EDATypeDescription.TEMPORAL
    else:
      return EDATypeDescription.GENERIC








from google.colab import auth
from google.cloud import bigquery
from google.cloud.bigquery import SchemaField
import pandas as pd # Opcional, para mostrarlo de forma más bonita

# Autentica tu usuario. Al ejecutar esto, aparecerá una ventana emergente
# para que inicies sesión con tu cuenta de Google y concedas permisos.
auth.authenticate_user()
print('¡Autenticación exitosa!')

# Connect the client for one and only one time

# MAYBE A SECRECT SECTION
PROJECT_ID : str = 'cpl-corp-mpd-prod-01082025'
DATASET_ID : str = 'cpl-corp-mpd-prod-01082025.mlops_v01_shared'
DATASET_NAME : str = 'mlops_v01_shared'
# Inicializa el cliente de BigQuery, que nos permitirá interactuar con el servicio.
try:
    client = bigquery.Client(project=PROJECT_ID)
    print(f"Cliente de BigQuery inicializado para el proyecto: '{PROJECT_ID}'")
except Exception as e:
    print(f"Error al inicializar el cliente: {e}")
    # Detener ejecución si hay un error aquí


from os import error
from functools import reduce
import pandas as pd
import google

# Reemplaza 'nombre_del_dataset' con el nombre del dataset que quieres analizar.
dataset_id = 'cpl-corp-mpd-prod-01082025.mlops_v01_shared'


tables = client.list_tables(dataset_id)
elements_ : list[list[dict]] = []

#Create Query section
# count hte number of columns and elemetns


tables_with_schema = []
# for table in [next(tables)]:
for table in tables:

  table_ref = DATASET_ID + '.' + table.table_id

  full_table = client.get_table(table_ref)
  list_scema_field : list[SchemaField] = full_table.schema

  stats_table = {}
  error_table = {}
  fields = []
  print("====================================================")
  query = f"""
WITH
-- Paso 1: Obtener el número de columnas de los metadatos
ColumnInfo AS (
  SELECT
    COUNT(*) AS numero_de_columnas
  FROM
    `{DATASET_ID}.INFORMATION_SCHEMA.COLUMNS`
  WHERE
    table_name = '{table.table_id}'
),
-- Paso 2: Obtener el número de filas de la tabla
RowInfo AS (
  SELECT
    COUNT(*) AS numero_de_filas
  FROM
    `{DATASET_ID}.{table.table_id}`
),
CreationTime AS (
  SELECT
    creation_time
    FROM
    `{DATASET_ID}.INFORMATION_SCHEMA.TABLES`
  WHERE
    table_name = '{table.table_id}'
)
-- Paso 3: Combinar los resultados para el reporte final
SELECT
ColumnInfo.numero_de_columnas as num_columns,
RowInfo.numero_de_filas as num_rows,
CreationTime.creation_time as creation_time
FROM
ColumnInfo,
RowInfo,
CreationTime;
"""
  try:
    tables_stats_df = client.query(query).to_dataframe()
    stats_table["accessibility"] = True
    stats_table["num_columns"] = int(tables_stats_df["num_columns"][0])
    stats_table["num_rows"] = int(tables_stats_df["num_rows"][0])
    creation_date_utc = tables_stats_df["creation_time"][0]
    creation_date_to_mx_time = creation_date_utc.tz_convert("America/Mexico_City")
    stats_table["creation_time"] = creation_date_to_mx_time.strftime("%Y-%m-%d %H:%M:%S")
  # except google.api_core.exceptions.BadRequest as e:
  #   error_table["missing_permition"] = e.message
  #   stats_table["num_columns"] = None
  #   stats_table["num_rows"] = None
  #   stats_table["accessibility"] = False
  #   continue
  # except google.api_core.exceptions.Forbidden as e:
  #   error_table["missing_query_filter"] = e.message
  #   stats_table["accessibility"] = False
  #   stats_table["num_columns"] = None
  #   stats_table["num_rows"] = None
  #   continue
  except Exception as e:
    print("AYUDA WEY")
    print(table_ref)
    print(e)
    error_table["missing_query"] = e.message
    stats_table["accessibility"] = False
    stats_table["num_columns"] = None
    stats_table["num_rows"] = None
  print(stats_table)
  print(error_table)
  print(f"Nombre de la tabla: {full_table.table_id}")
  # print(type(full_table.schema[0]))
  if not full_table.schema:
        print("  (La tabla no tiene un esquema definido o está vacía)")
  else:
      elements_table = []
      for field in full_table.schema:
          data_field: dict = field.to_api_repr()
          #Add table name
          data_field["table_ref"] = table_ref
          type_field = DescriptionOperator.select_description_object(field)
          despcription_wrapper: FieldEDADescription = DescriptionOperator.wrapper_select_description_object(field, type_field)

          eda_schema_field = EDASchemaField(
              name=field.name,
              field_type=field.field_type,
              mode=field.mode,
              eda_description=despcription_wrapper,
              description=field.description
          )
          elements_table.append(eda_schema_field)

  schema_table = SchemaTable(
      name=table.table_id,
      fields=elements_table,
      error=error_table,
      stats=stats_table,
      table_id=table_ref
  )

  # print(schema_table)

  tables_with_schema.append(schema_table)


¡Autenticación exitosa!
Cliente de BigQuery inicializado para el proyecto: 'cpl-corp-mpd-prod-01082025'
{'accessibility': True, 'num_columns': 2, 'num_rows': 43, 'creation_time': '2026-04-24 11:57:48'}
{}
Nombre de la tabla: vw_abonos_prestamos_foliosssffsoftlaunch
AYUDA WEY
cpl-corp-mpd-prod-01082025.mlops_v01_shared.vw_analytics_abono_coppel_trafico_bancoppel_app
400 Cannot query over table '`cpl-corp-mpd-prod-01082025`.mlops_v01_shared.vw_analytics_abono_coppel_trafico_bancoppel_app' without a filter over column(s) 'fec_FechaVisita' that can be used for partition elimination; reason: invalidQuery, location: query, message: Cannot query over table '`cpl-corp-mpd-prod-01082025`.mlops_v01_shared.vw_analytics_abono_coppel_trafico_bancoppel_app' without a filter over column(s) 'fec_FechaVisita' that can be used for partition elimination

Location: US
Job ID: 547da3fe-5887-4fee-b6dc-e7fe7358d91c

{'accessibility': False, 'num_columns': None, 'num_rows': None}
{'missing_query': "Cannot que

# Step 2. Get the columns for temprary column
tables_with_schema

In [ ]:
# Change simple TableWithProcess



import re
class ExtraQueryForFistDeniedSelect:
  def get_first_missing_column_for_correct_query(str_error: str) -> Optional[str]:
    # text = "over column(s) 'fec_FechaCorte'"

    # The regex pattern
    pattern = r"over column\(s\)\s+'([^']+)'"
    # Search for the match
    match = re.search(pattern, str_error)

    if match:
        column_name = match.group(1)
        return column_name
    else:
        return None


#### CLASS MODELS

class TableWithProcessDict:
  def __init__(self, table: SchemaTable):
    self.table = table
    self.process = {}
    self.metadata = {}

#Map the first process error:
tables_with_process = list(map(lambda x: TableWithProcessDict(x), tables_with_schema))

def frist_process_with_error(table: TableWithProcessDict) -> TableWithProcessDict:
  if table.table.error == {}:
    table.process["first_process_log"] = "No errors"
    return table
  else:
    error_str : str = table.table.error["missing_query"]
    table.process["first_process_log"] = error_str
    return table

tables_with_process = list(map(frist_process_with_error, tables_with_process))

### BEGIN SECOND PROCESS
def second_process_get_column_for_periodicity(table_with_process: TableWithProcessDict) -> TableWithProcessDict:
  if table_with_process.process["first_process_log"] == "No errors":
    return table_with_process
  else:
    error_str : str = table_with_process.process["first_process_log"]
    if "reason: invalidQuery, location: query" in error_str:
      periodicity_column_name = ExtraQueryForFistDeniedSelect.get_first_missing_column_for_correct_query(error_str)
      table_with_process.metadata["periodicity_column_name"] = periodicity_column_name
      return table_with_process
    else:
      return table_with_process

tables_with_process = list(map(second_process_get_column_for_periodicity, tables_with_process))

# STEP 3. Recalculate table sum for tables with missing periodicity column

In [ ]:
#Reprocess for missing query

def update_stats_data_table_with_accionable_error(table: SchemaTable) -> SchemaTable:
  if table.error == {}:
    return table
  else:
    error_str : str = table.error["missing_query"]
    if "reason: accessDenied" in error_str:
      return table
    elif "reason: invalidQuery, location: query" in error_str:
      # Retry for stast data, and just that.

      missing_column_name = ExtraQueryForFistDeniedSelect.get_first_missing_column_for_correct_query(error_str)

      query = f"""
              WITH
              -- Paso 1: Obtener el número de columnas de los metadatos
              ColumnInfo AS (
                SELECT
                  COUNT(*) AS numero_de_columnas
                FROM
                  `{DATASET_ID}.INFORMATION_SCHEMA.COLUMNS`
                WHERE
                  table_name = '{table.name}'
              ),
              -- Paso 2: Obtener el número de filas de la tabla
              RowInfo AS (
                SELECT
                  COUNT(*) AS numero_de_filas
                FROM
                  `{DATASET_ID}.{table.name}`
                WHERE {missing_column_name} > (CURRENT_DATE - INTERVAL 1 year)
              ),
              CreationTime AS (
                SELECT
                  creation_time
                  FROM
                  `{DATASET_ID}.INFORMATION_SCHEMA.TABLES`
                WHERE
                  table_name = '{table.name}'
              )
              -- Paso 3: Combinar los resultados para el reporte final
              SELECT
              ColumnInfo.numero_de_columnas as num_columns,
              RowInfo.numero_de_filas as num_rows,
              CreationTime.creation_time as creation_time
              FROM
              ColumnInfo,
              RowInfo,
              CreationTime;
              """
      try:
        tables_stats_df = client.query(query).to_dataframe()
        #table.stats["accessibility"] = True
        table.stats["num_columns"] = int(tables_stats_df["num_columns"][0])
        table.stats["num_rows"] = int(tables_stats_df["num_rows"][0])
        creation_date_utc = tables_stats_df["creation_time"][0]
        creation_date_to_mx_time = creation_date_utc.tz_convert("America/Mexico_City")
        table.stats["creation_time"] = creation_date_to_mx_time.strftime("%Y-%m-%d %H:%M:%S")
        return table
      except Exception as e:
        print("AYUDA WEY")
        print(table_ref)
        print(e)
        error_table["missing_query"] = e.message
        table.stats["accessibility"] = False
        table.stats["num_columns"] = None
        table.stats["num_rows"] = None
        return table
    else:
      return table
    #Type error:


#tables_with_schema = list(map(update_stats_data_table_with_accionable_error, tables_with_schema))
# tables_with_schema
def third_stage_update_stats_data_table_with_accionable_error(table: TableWithProcessDict) -> TableWithProcessDict:
  table_schema = table.table
  updated_schema = update_stats_data_table_with_accionable_error(table_schema)
  table.table = updated_schema
  return table

#tables_with_process = list(map(update_stats_data_table_with_accionable_error, tables_with_process))
tables_with_process = list(map(third_stage_update_stats_data_table_with_accionable_error, tables_with_process))

# STEP 4. Calculate periodicity for missin columns

In [ ]:
from enum import Enum

class EDAPeriodicity(Enum):
  ANUAL = "ANUAL"
  SEMESTRAL = "SEMESTRAL"
  TRIMESTRAL = "TRIMESTRAL"
  MENSUAL = "MENSUAL"
  SEMANAL = "SEMANAL"
  DIARIO = "DIARIO"
  TRANSACIONAL = "TRANSACIONAL"
  SIN_PERIODICIDAD = "SIN_PERIODICIDAD"

class PeriodicityOperator:
  def calculate_periodicity_by_year_fraction(year_fraction: float) -> EDAPeriodicity:
    if 0 < year_fraction <= 1 / 365:
      return EDAPeriodicity.ANUAL
    elif 1 / 365 < year_fraction <= 2 / 365 :
      return EDAPeriodicity.SEMESTRAL
    elif 2 / 365 < year_fraction <= 4 / 365 :
      return EDAPeriodicity.TRIMESTRAL
    elif 4 / 365 < year_fraction <= 12 / 365 :
      return EDAPeriodicity.MENSUAL
    elif 12 / 365 < year_fraction <= 52 / 365 :
      return EDAPeriodicity.SEMANAL
    elif 52 / 365 < year_fraction <= 1 :
      return EDAPeriodicity.DIARIO
    elif 1 < year_fraction:
      return EDAPeriodicity.TRANSACIONAL
    else:
      return EDAPeriodicity.SIN_PERIODICIDAD


def stage_four_calculate_periodicity_for_missing_column(table: TableWithProcessDict) -> TableWithProcessDict:
  if table.metadata == {}:
    return table
  else:
    missing_columns_for_query = table.metadata["periodicity_column_name"]
    counting_query = f"""
          SELECT
            COUNT( DISTINCT {missing_columns_for_query}) as count_distinct_values
          FROM
            `{table.table.table_id}`
          WHERE
            {missing_columns_for_query} > (CURRENT_DATE - INTERVAL 1 YEAR);"""
    result = float(client.query(counting_query).to_dataframe()["count_distinct_values"][0]) / 365
    periodicity = PeriodicityOperator.calculate_periodicity_by_year_fraction(result)
    table.metadata["periodicity"] = periodicity
    return table

tables_with_process = list(map(stage_four_calculate_periodicity_for_missing_column, tables_with_process))

# Step 4.5 = Normalice values for missing periodicit calculation


In [ ]:
def stage_4_and_half_add_periodicity_value_in_stats(table: TableWithProcessDict) -> TableWithProcessDict:
  if table.metadata == {}:
    return table
  else:
    periodicity = table.metadata["periodicity"]
    table.table.stats["periodicity"] = periodicity
    return table

tables_with_process = list(map(stage_4_and_half_add_periodicity_value_in_stats, tables_with_process))

# STEP 5: Calculate periodicity for non missing columns


In [ ]:
#foo = list(filter(lambda x: x.metadata == {}, tables_with_process))

# Note: Create this query with no limit of time
def stage_five_set_periodicity_column_name_for_non_missing_column(table: TableWithProcessDict) -> TableWithProcessDict:
  if table.metadata == {}:
    table_contains_field_with_periodicity = any(f.eda_description.type_description == EDATypeDescription.TEMPORAL for f in table.table.fields)
    if table_contains_field_with_periodicity:
      print(table.table.name)
      # get_temporal_field_that_contains_word_actualizacion = lambda x: next(filter(lambda y: y.eda_description.type_description == EDATypeDescription.TEMPORAL, x.fields), None)
      temporal_fields = list(filter(lambda x: x.eda_description.type_description == EDATypeDescription.TEMPORAL, table.table.fields))

      # print(len(temporal_fields))
      #print(table.table.fields)
      if len(temporal_fields) == 1:
        # print(temporal_fields[0].name)
        table.metadata["periodicity_column_name"] = temporal_fields[0].name
      else:
        print([f.name for f in temporal_fields])
        possible_names= [f.name for f in temporal_fields]
        #fec_FechaFacturacion
        #particion
        #fec_Actualizacion
        #fec_fechacorte
        #fec_Movimiento
        #fec_FechaSql
        #fec_presolicitud
        prefer_words = ["facturacion", "particion", "actualizacion", "corte", "movimiento", "fechasql", "solicitud", "fec"]
        sorted_possibles_names_by_apperance_of_preder_words = sorted(possible_names, key=lambda x: sum(x.lower().count(y) for y in prefer_words), reverse=True)
        table.metadata["periodicity_column_name"] = sorted_possibles_names_by_apperance_of_preder_words[0]
      return table
    else:
      table.metadata["periodicity_column_name"] = None
      table.metadata["periodicity"] = EDAPeriodicity.SIN_PERIODICIDAD
      return table
  else:
    return table

tables_with_process = list(map(stage_five_set_periodicity_column_name_for_non_missing_column, tables_with_process))

vw_analytics_atribucion_pagare_ssff_app
['fec_Evento', 'fec_FirstOpen', 'fec_EventoHora']
vw_analytics_entradas_abonos_app
vw_analytics_funnel_prestamos_app
vw_analytics_funnel_prestamos_com
vw_analytics_landing_invitado_prestamos_com
['fec_FechaVisita', 'fec_ParticionDomo']
vw_analytics_usuarios_unicos_web_vista
['num_Anio', 'num_Semana', 'fec_FechaInicio', 'fec_FechaFin']
vw_cat_estructuratiendas
vw_coppelpayinfra_movimientos_coppelpay
['fec_UltimaVenta', 'fec_HoraInicio', 'fec_HoraFin', 'fec_FechaCorte']
vw_ctl_colaborador
vw_formalizacion_dud
vw_funnel_candidatos_abonos
vw_funneltrafico_appcoppel_prestamos
vw_inflacion
['fec_Date', 'fec_ParticionDomo']
vw_mae_abonos_aprobacion_general
vw_mae_abonos_diario
['fec_Movimiento', 'fec_FraudeAjuste', 'fec_PrimeraCompra', 'fec_Tienda', 'fec_PrimerAbono', 'num_HoraMovimiento']
vw_mae_abonosgrupocoppel
vw_mae_actividad_clientescredito
vw_mae_buildingblockpresolicitudes
vw_mae_catalogoinflacion_ar
['fec_FechaMes', 'num_InflacionMes']
vw_mae_c

# STEP 6.1: Get table name


In [ ]:
def step_6_get_table_name(table: TableWithProcessDict) -> TableWithProcessDict:
  table_id_for_reference = table.table.table_id
  table.metadata["table_page_name"] = table_id_for_reference
  return table

tables_with_process = list(map(step_6_get_table_name, tables_with_process))

#STEP 6.2 Generate table first table sum

In [ ]:
def step_6_get_table_sum_for_simple_statistic(table: TableWithProcessDict) -> TableWithProcessDict:
  stats = table.table.stats
  df_summary_table = pd.DataFrame(stats, index=[0])
  print(table.metadata["table_page_name"])


  df_summary_table_readable_names_columns = {
      "num_columns" : "Número de columnas",
      "num_rows" : "Número de filas",
      "creation_time" : "Fecha de creación",
      "accessibility" : "Accesibilidad",
      "periodicity" : "Periodicidad"
  }
  df_summary_table.rename(columns=df_summary_table_readable_names_columns, inplace=True)
  # if periodicity column name appeaars, added a new column for the column_name_periodicity
  if "periodicity_column_name" in table.metadata:
    df_summary_table["Columna de periodicidad"] = table.metadata["periodicity_column_name"]

  table.metadata["table_page_table_sum"] = df_summary_table
  return table

tables_with_process = list(map(step_6_get_table_sum_for_simple_statistic, tables_with_process))

cpl-corp-mpd-prod-01082025.mlops_v01_shared.vw_abonos_prestamos_foliosssffsoftlaunch
cpl-corp-mpd-prod-01082025.mlops_v01_shared.vw_analytics_abono_coppel_trafico_bancoppel_app
cpl-corp-mpd-prod-01082025.mlops_v01_shared.vw_analytics_atribucion_n2_eventos_colaterales_ssff_app
cpl-corp-mpd-prod-01082025.mlops_v01_shared.vw_analytics_atribucion_n2_ssff_app
cpl-corp-mpd-prod-01082025.mlops_v01_shared.vw_analytics_atribucion_pagare_ssff_app
cpl-corp-mpd-prod-01082025.mlops_v01_shared.vw_analytics_ayuda_promotor_n2_ssff_app
cpl-corp-mpd-prod-01082025.mlops_v01_shared.vw_analytics_entradas_abonos_app
cpl-corp-mpd-prod-01082025.mlops_v01_shared.vw_analytics_funnel_abonos_app
cpl-corp-mpd-prod-01082025.mlops_v01_shared.vw_analytics_funnel_abonos_com
cpl-corp-mpd-prod-01082025.mlops_v01_shared.vw_analytics_funnel_prestamos_app
cpl-corp-mpd-prod-01082025.mlops_v01_shared.vw_analytics_funnel_prestamos_com
cpl-corp-mpd-prod-01082025.mlops_v01_shared.vw_analytics_landing_invitado_prestamos_com
cpl-

# STEP 6.3 Generate column values

In [ ]:
def step_6_3_get_table_columns_info(table: TableWithProcessDict) -> TableWithProcessDict:
  is_valid_table_to_calculate = table.table.stats["accessibility"]

  print(table.table.table_id)


  if not is_valid_table_to_calculate:
    fields = table.table.fields
    df_field_information = pd.DataFrame({"Nombre columna": [f.name for f in fields],
                                         "Tipo": [f.eda_description.type_description.value for f in fields]})
    table.metadata["table_page_columns_info"] = df_field_information
    return table
  else:
    try:
      query_data_columns = table.table.generate_query()
      df_columns_info = client.query_and_wait(query_data_columns, wait_timeout = 90).to_dataframe()
      error_message = " "

      #df_columns_info = client.query(query_data_columns).to_dataframe()
      column_names = [f.name for f in table.table.fields]

      list_data_columns = []
      for field in table.table.fields:
        name = field.name
        type_field = field.eda_description.type_description.value
        info_column = {}
        info_column["name"] = name
        info_column["type"] = type_field
        valid_calculations_columns = filter(lambda x: x.startswith(name), df_columns_info.columns)
        for valid_column in valid_calculations_columns:
          value = df_columns_info[valid_column][0]

          if "missing_values" in valid_column:
            value = int(value)
            info_column["missing_values"] = value
          elif "count_values" in valid_column:
            value = int(value)
            info_column["count_values"] = value
          elif "min_value" in valid_column:
            # value = float(value)
            info_column["min_value"] = value
          elif "max_value" in valid_column:
            # value = float(value)
            info_column["max_value"] = value
          elif "unique_values" in valid_column:
            value = int(value)
            info_column["unique_values"] = value
          elif "std_value" in valid_column:
            value = float(value)
            info_column["std_value"] = value
          elif "mean_value" in valid_column:
            value = float(value)
            info_column["mean_value"] = value

        list_data_columns.append(info_column)

      df_columns_info = pd.DataFrame(list_data_columns)
      #Calculate percetaneje of missing values
      df_columns_info["missing_values_percentage"] = df_columns_info["missing_values"] / (df_columns_info["count_values"] + df_columns_info["missing_values"])
      #Saved with only two decimales
      df_columns_info["missing_values_percentage"] = df_columns_info["missing_values_percentage"].apply(lambda x: round(x, 2))
      # Replace names Columns
      df_columns_info_table_readable_names_columns = {
          "name" : "Nombre columna",
          "type" : "Tipo",
          "missing_values" : "Valores faltantes",
          "count_values" : "Conteo de valores",
          "min_value" : "Valor mínimo",
          "max_value" : "Valor máximo",
          "unique_values" : "Valores únicos",
          "std_value" : "Desviación estándar",
          "mean_value" : "Media",
          "missing_values_percentage" : "Porcentaje de valores faltantes"
      }
      df_columns_info.rename(columns=df_columns_info_table_readable_names_columns, inplace=True)
      #Saved with another order of columns
      #df_columns_info = df_columns_info[["Nombre columna", "Tipo", "Conteo de valores", "Valores faltantes", "Porcentaje de valores faltantes", "Valor mínimo", "Media", "Desviación estándar", "Valor máximo", "Valores únicos"]]

      table.metadata["table_page_columns_info"] = df_columns_info
      return table
    except Exception as e:
      error_message = e.message
      table.metadata["table_page_columns_info"] = None
      return table

tables_with_process = list(map(step_6_3_get_table_columns_info, tables_with_process))

cpl-corp-mpd-prod-01082025.mlops_v01_shared.vw_abonos_prestamos_foliosssffsoftlaunch
cpl-corp-mpd-prod-01082025.mlops_v01_shared.vw_analytics_abono_coppel_trafico_bancoppel_app
cpl-corp-mpd-prod-01082025.mlops_v01_shared.vw_analytics_atribucion_n2_eventos_colaterales_ssff_app
cpl-corp-mpd-prod-01082025.mlops_v01_shared.vw_analytics_atribucion_n2_ssff_app
cpl-corp-mpd-prod-01082025.mlops_v01_shared.vw_analytics_atribucion_pagare_ssff_app
cpl-corp-mpd-prod-01082025.mlops_v01_shared.vw_analytics_ayuda_promotor_n2_ssff_app
cpl-corp-mpd-prod-01082025.mlops_v01_shared.vw_analytics_entradas_abonos_app
cpl-corp-mpd-prod-01082025.mlops_v01_shared.vw_analytics_funnel_abonos_app
cpl-corp-mpd-prod-01082025.mlops_v01_shared.vw_analytics_funnel_abonos_com
cpl-corp-mpd-prod-01082025.mlops_v01_shared.vw_analytics_funnel_prestamos_app
cpl-corp-mpd-prod-01082025.mlops_v01_shared.vw_analytics_funnel_prestamos_com
cpl-corp-mpd-prod-01082025.mlops_v01_shared.vw_analytics_landing_invitado_prestamos_com
cpl-

# SETP 6.4 Add error description

In [ ]:
def set_6_4_add_error_description(table: TableWithProcessDict) -> TableWithProcessDict:
  errors : dict = table.table.error
  if "missing_query" in errors:
    error_txt = errors["missing_query"]
    if "reason: accessDenied" in error_txt:
      table.metadata["table_page_error"] = error_txt
      return table
    else:
      table.metadata["table_page_error"] = None
      return table
  else:
    table.metadata["table_page_error"] = None
    return table

tables_with_process = list(map(set_6_4_add_error_description, tables_with_process))

# STEP 8: Erros for paarticular cases



In [ ]:
def retry_metrics_for_missing_column_name(table: TableWithProcessDict) -> TableWithProcessDict:
  special_tables = ["vw_his_mae_articulos",
                  "vw_mae_aprobacionabonosprueba",
                  "vw_his_mae_entregas",
                  "vw_mae_abonos",
                  "vw_mae_aprobacion",
                  "vw_mae_aprobacionabonos",
                  "vw_mae_diferenciaprecioscompetidores",
                  "vw_mae_ordenes",
                  "vw_mae_prestamos"]
  table_name = table.table.name.split(".")[-1].lower()
  print("===========================")
  print(table.table.table_id)
  if table.table.error == {}:
    return table
  else:
    if "missing_query" in table.table.error:
      error_str : str = table.table.error["missing_query"]
      if "reason: accessDenied" in error_str:
        return table
      elif table_name in special_tables:
        return table
      elif "reason: invalidQuery, location: query" in error_str:
        try:
          query_data_columns = table.table.generate_query()
          #Extract columns from error
          missing_column_name = ExtraQueryForFistDeniedSelect.get_first_missing_column_for_correct_query(error_str)
          #Add where clause into query_data_columns
          query_data_columns = query_data_columns + f" WHERE {missing_column_name} > (CURRENT_DATE - INTERVAL 1 YEAR);"
          print("QUERY JOB")
          print(query_data_columns)
          df_columns_info = client.query(query_data_columns).to_dataframe()

          #df_columns_info = client.query(query_data_columns).to_dataframe()
          column_names = [f.name for f in table.table.fields]

          list_data_columns = []
          for field in table.table.fields:
            name = field.name
            type_field = field.eda_description.type_description.value
            info_column = {}
            info_column["name"] = name
            info_column["type"] = type_field
            valid_calculations_columns = filter(lambda x: x.startswith(name), df_columns_info.columns)
            for valid_column in valid_calculations_columns:
              value = df_columns_info[valid_column][0]


              if "missing_values" in valid_column:
                value = int(value)
                info_column["missing_values"] = value
              elif "count_values" in valid_column:
                value = int(value)
                info_column["count_values"] = value
              elif "min_value" in valid_column:
                # value = float(value)
                info_column["min_value"] = value
              elif "max_value" in valid_column:
                # value = float(value)
                info_column["max_value"] = value
              elif "unique_values" in valid_column:
                value = int(value)
                info_column["unique_values"] = value
              elif "std_value" in valid_column:
                value = float(value)
                info_column["std_value"] = value
              elif "mean_value" in valid_column:
                value = float(value)
                info_column["mean_value"] = value

            list_data_columns.append(info_column)

          df_columns_info = pd.DataFrame(list_data_columns)
          #Calculate percetaneje of missing values
          df_columns_info["missing_values_percentage"] = df_columns_info["missing_values"] / (df_columns_info["count_values"] + df_columns_info["missing_values"])
          #Saved with only two decimales
          df_columns_info["missing_values_percentage"] = df_columns_info["missing_values_percentage"].apply(lambda x: round(x, 2))
          # Replace names Columns
          df_columns_info_table_readable_names_columns = {
              "name" : "Nombre columna",
              "type" : "Tipo",
              "missing_values" : "Valores faltantes",
              "count_values" : "Conteo de valores",
              "min_value" : "Valor mínimo",
              "max_value" : "Valor máximo",
              "unique_values" : "Valores únicos",
              "std_value" : "Desviación estándar",
              "mean_value" : "Media",
              "missing_values_percentage" : "Porcentaje de valores faltantes"
          }
          df_columns_info.rename(columns=df_columns_info_table_readable_names_columns, inplace=True)
          table.metadata["table_page_columns_info"] = df_columns_info
          return table
        except Exception as e:
          print(e)
          return table
      else:
        return table

    else:
      return table

tables_with_process = list(map(retry_metrics_for_missing_column_name, tables_with_process))

cpl-corp-mpd-prod-01082025.mlops_v01_shared.vw_abonos_prestamos_foliosssffsoftlaunch
cpl-corp-mpd-prod-01082025.mlops_v01_shared.vw_analytics_abono_coppel_trafico_bancoppel_app
QUERY JOB

      SELECT
        SUM(CASE WHEN fec_FechaVisita IS NULL THEN 1 ELSE 0 END) as fec_FechaVisita_missing_values,
MIN(fec_FechaVisita) as fec_FechaVisita_min_value,
MAX(fec_FechaVisita) as fec_FechaVisita_max_value,
SUM(CASE WHEN des_PaginaVisita IS NULL THEN 1 ELSE 0 END) as des_PaginaVisita_missing_values,
COUNT(des_PaginaVisita) as des_PaginaVisita_count_values,
COUNT(DISTINCT des_PaginaVisita) as des_PaginaVisita_unique_values,
SUM(CASE WHEN des_Plataforma IS NULL THEN 1 ELSE 0 END) as des_Plataforma_missing_values,
COUNT(des_Plataforma) as des_Plataforma_count_values,
COUNT(DISTINCT des_Plataforma) as des_Plataforma_unique_values,
SUM(CASE WHEN num_Visitas IS NULL THEN 1 ELSE 0 END) as num_Visitas_missing_values,
COUNT(num_Visitas) as num_Visitas_count_values,
SUM(CASE WHEN num_VisitasUnicas IS NU

## STEP 7.1 calculate temporalidad por otros casos.

In [ ]:
def step_7_1_caalculate_temporalidad_for_other_cases(table: TableWithProcessDict) -> TableWithProcessDict:
  if table.table.error == {} and table.metadata["periodicity_column_name"] != None:
    missing_columns_for_query = table.metadata["periodicity_column_name"]
    counting_query = f"""
          SELECT
            COUNT( DISTINCT {missing_columns_for_query}) as count_distinct_values
          FROM
            `{table.table.table_id}`
          WHERE
            {missing_columns_for_query} > (CURRENT_DATE - INTERVAL 1 YEAR);"""
    print("===========================")
    print(table.table.table_id)
    print(counting_query)
    try:
      result = float(client.query(counting_query).to_dataframe()["count_distinct_values"][0]) / 365
      periodicity = PeriodicityOperator.calculate_periodicity_by_year_fraction(result)
      df_to_update = table.metadata["table_page_table_sum"]
      df_to_update["Periodicidad"] = periodicity
      table.metadata["table_page_table_sum"] = df_to_update
      return table
    except Exception as e:
      print(e)
      return table
  else:
    return table


tables_with_process = list(map(step_7_1_caalculate_temporalidad_for_other_cases, tables_with_process))

cpl-corp-mpd-prod-01082025.mlops_v01_shared.vw_analytics_atribucion_pagare_ssff_app

          SELECT
            COUNT( DISTINCT fec_Evento) as count_distinct_values
          FROM
            `cpl-corp-mpd-prod-01082025.mlops_v01_shared.vw_analytics_atribucion_pagare_ssff_app`
          WHERE
            fec_Evento > (CURRENT_DATE - INTERVAL 1 YEAR);
cpl-corp-mpd-prod-01082025.mlops_v01_shared.vw_analytics_entradas_abonos_app

          SELECT
            COUNT( DISTINCT date) as count_distinct_values
          FROM
            `cpl-corp-mpd-prod-01082025.mlops_v01_shared.vw_analytics_entradas_abonos_app`
          WHERE
            date > (CURRENT_DATE - INTERVAL 1 YEAR);
cpl-corp-mpd-prod-01082025.mlops_v01_shared.vw_analytics_funnel_prestamos_app

          SELECT
            COUNT( DISTINCT des_FechaVisita) as count_distinct_values
          FROM
            `cpl-corp-mpd-prod-01082025.mlops_v01_shared.vw_analytics_funnel_prestamos_app`
          WHERE
            des_FechaVisita

In [ ]:
## STEP 7.2 Recalculate number of rows

def recalculate_number_of_rows(table: TableWithProcessDict) -> TableWithProcessDict:
  query_alias = "EDAQueryForCENIC"
  table_reference_sql = table.table.table_id
  sql_query = f"""WITH {query_alias} AS (
                      SELECT
                          COUNT(*) AS num_rows
                      FROM
                          {table_reference_sql}
                  )
                  SELECT * FROM {query_alias}"""
  print("============================")
  print(table.table.table_id)
  try:
    df = client.query(sql_query).to_dataframe()
    print(sql_query)
    num_rows = int(df["num_rows"][0])
    df_to_update = table.metadata["table_page_table_sum"]
    df_to_update["Número de filas"] = num_rows
    table.metadata["table_page_table_sum"] = df_to_update
    return table
  except Exception as e:
    error_str = str(e)
    if "reason: invalidQuery, location: query" in error_str:
      periodicity_column_name = ExtraQueryForFistDeniedSelect.get_first_missing_column_for_correct_query(error_str)
      sql_query = f"""WITH {query_alias} AS (
                          SELECT
                              COUNT(*) AS num_rows
                          FROM
                              {table_reference_sql}
                          WHERE
                              {periodicity_column_name} < CURRENT_DATE
                          )
                          SELECT * FROM {query_alias}
                          """
      print(sql_query)
      df = client.query(sql_query).to_dataframe()
      num_rows = int(df["num_rows"][0])
      df_to_update = table.metadata["table_page_table_sum"]
      df_to_update["Número de filas"] = num_rows
      table.metadata["table_page_table_sum"] = df_to_update
      return table
    else:
      return table
  return table

tables_with_process = list(map(recalculate_number_of_rows, tables_with_process))

cpl-corp-mpd-prod-01082025.mlops_v01_shared.vw_abonos_prestamos_foliosssffsoftlaunch
WITH EDAQueryForCENIC AS (
                      SELECT
                          COUNT(*) AS num_rows
                      FROM
                          cpl-corp-mpd-prod-01082025.mlops_v01_shared.vw_abonos_prestamos_foliosssffsoftlaunch
                  )
                  SELECT * FROM EDAQueryForCENIC
cpl-corp-mpd-prod-01082025.mlops_v01_shared.vw_analytics_abono_coppel_trafico_bancoppel_app
WITH EDAQueryForCENIC AS (
                          SELECT
                              COUNT(*) AS num_rows
                          FROM
                              cpl-corp-mpd-prod-01082025.mlops_v01_shared.vw_analytics_abono_coppel_trafico_bancoppel_app
                          WHERE
                              fec_FechaVisita < CURRENT_DATE
                          )
                          SELECT * FROM EDAQueryForCENIC
                          
cpl-corp-mpd-prod-01082025.mlops_v01_shared.

# STEP 8.0: Stats before excel


In [ ]:
def print_stats_before_excel(table: TableWithProcessDict) -> TableWithProcessDict:
  print("===========================")
  print(table.table.table_id)
  print(table.table.stats)
  print(table.metadata["table_page_table_sum"])

  return table

print_stats_before_excel = list(map(print_stats_before_excel, tables_with_process))

cpl-corp-mpd-prod-01082025.mlops_v01_shared.vw_abonos_prestamos_foliosssffsoftlaunch
{'accessibility': True, 'num_columns': 2, 'num_rows': 43, 'creation_time': '2026-04-24 11:57:48'}
   Accesibilidad  Número de columnas  Número de filas    Fecha de creación  \
0           True                   2               43  2026-04-24 11:57:48   

  Columna de periodicidad                                     Tabla  \
0                    None  vw_abonos_prestamos_foliosssffsoftlaunch   

   Porcentaje de valores faltantes  
0                         0.372093  
cpl-corp-mpd-prod-01082025.mlops_v01_shared.vw_analytics_abono_coppel_trafico_bancoppel_app
{'accessibility': False, 'num_columns': 10, 'num_rows': 31, 'creation_time': '2026-04-30 13:52:12', 'periodicity': <EDAPeriodicity.MENSUAL: 'MENSUAL'>}
   Accesibilidad  Número de columnas  Número de filas    Fecha de creación  \
0           True                  10          3116901  2026-04-30 13:52:12   

  Periodicidad Columna de periodicidad  \


# STEP 8: Generate Excel based on LIST[TableWithProcess]

## Clean names and change errors for displaying

In [ ]:
def change_periodiciy_objects_to_text(table: TableWithProcessDict) -> TableWithProcessDict:
  df_to_update = table.metadata["table_page_table_sum"]
  if "Periodicidad" in df_to_update.columns:
    df_to_update["Periodicidad"] = df_to_update["Periodicidad"].apply(lambda x: x.value)
    table.metadata["table_page_table_sum"] = df_to_update
    return table
  else:
    return table


tables_with_process = list(map(change_periodiciy_objects_to_text, tables_with_process))

AttributeError: 'str' object has no attribute 'value'

In [ ]:
def valid_error_to_display_accessdenide(table: TableWithProcessDict) -> TableWithProcessDict:
  error = table.table.error

  if "missing_query" in error:
    error_str : str = error["missing_query"]
    if "reason: accessDenied" in error_str:
      table.metadata["table_page_error"] = error_str
      df_to_update = table.metadata["table_page_table_sum"]
      df_to_update["Accesibilidad"] = False
      table.metadata["table_page_table_sum"] = df_to_update
      # print(error_str)
      return table
    else:
      table.metadata["table_page_error"] = None
      df_to_update = table.metadata["table_page_table_sum"]
      df_to_update["Accesibilidad"] = True
      table.metadata["table_page_table_sum"] = df_to_update
      return table
  else:
    table.metadata["table_page_error"] = None
    df_to_update = table.metadata["table_page_table_sum"]
    df_to_update["Accesibilidad"] = True
    table.metadata["table_page_table_sum"] = df_to_update
    return table


tables_with_process = list(map(valid_error_to_display_accessdenide, tables_with_process))

In [ ]:
def valid_columns_info_to_display(table: TableWithProcessDict) -> TableWithProcessDict:
  print(table.metadata["table_page_columns_info"].columns)
  columnas_validas = ["Nombre columna",
                      "Tipo",
                      'Valor mínimo',
                      'Valor máximo',
                      'Porcentaje de valores faltantes']
  try:
    table.metadata["table_page_columns_info"] = table.metadata["table_page_columns_info"][columnas_validas]
    return table
  except KeyError:
    return table


tables_with_process = list(map(valid_columns_info_to_display, tables_with_process))

Index(['Nombre columna', 'Tipo', 'Valores faltantes', 'Valores únicos',
       'Conteo de valores', 'Porcentaje de valores faltantes'],
      dtype='object')
Index(['Nombre columna', 'Tipo', 'Valor mínimo', 'Valor máximo',
       'Porcentaje de valores faltantes'],
      dtype='object')
Index(['Nombre columna', 'Tipo', 'Valor mínimo', 'Valor máximo',
       'Porcentaje de valores faltantes'],
      dtype='object')
Index(['Nombre columna', 'Tipo', 'Valor mínimo', 'Valor máximo',
       'Porcentaje de valores faltantes'],
      dtype='object')
Index(['Nombre columna', 'Tipo', 'Valor mínimo', 'Valor máximo',
       'Porcentaje de valores faltantes'],
      dtype='object')
Index(['Nombre columna', 'Tipo', 'Valor mínimo', 'Valor máximo',
       'Porcentaje de valores faltantes'],
      dtype='object')
Index(['Nombre columna', 'Tipo', 'Valor mínimo', 'Valor máximo',
       'Porcentaje de valores faltantes'],
      dtype='object')
Index(['Nombre columna', 'Tipo', 'Valor mínimo', 'Valor máximo

In [ ]:
info_for_page = {}
for t in tables_with_process:

  name = t.metadata["table_page_name"]
  df_summ = t.metadata["table_page_table_sum"]
  df_col = t.metadata["table_page_columns_info"]
  error = t.metadata["table_page_error"]
  name_table = t.table.name.split(".")[-1]
  print(name)
  info_for_page[name_table] = {
      "table_ref": name,
      "table_sum": df_summ,
      "table_col": df_col,
      "error": error
  }

cpl-corp-mpd-prod-01082025.mlops_v01_shared.vw_abonos_prestamos_foliosssffsoftlaunch
cpl-corp-mpd-prod-01082025.mlops_v01_shared.vw_analytics_abono_coppel_trafico_bancoppel_app
cpl-corp-mpd-prod-01082025.mlops_v01_shared.vw_analytics_atribucion_n2_eventos_colaterales_ssff_app
cpl-corp-mpd-prod-01082025.mlops_v01_shared.vw_analytics_atribucion_n2_ssff_app
cpl-corp-mpd-prod-01082025.mlops_v01_shared.vw_analytics_atribucion_pagare_ssff_app
cpl-corp-mpd-prod-01082025.mlops_v01_shared.vw_analytics_ayuda_promotor_n2_ssff_app
cpl-corp-mpd-prod-01082025.mlops_v01_shared.vw_analytics_entradas_abonos_app
cpl-corp-mpd-prod-01082025.mlops_v01_shared.vw_analytics_funnel_abonos_app
cpl-corp-mpd-prod-01082025.mlops_v01_shared.vw_analytics_funnel_abonos_com
cpl-corp-mpd-prod-01082025.mlops_v01_shared.vw_analytics_funnel_prestamos_app
cpl-corp-mpd-prod-01082025.mlops_v01_shared.vw_analytics_funnel_prestamos_com
cpl-corp-mpd-prod-01082025.mlops_v01_shared.vw_analytics_landing_invitado_prestamos_com
cpl-

## Excel section

In [ ]:
import pandas as pd
import datetime

# Assuming info_for_page is already populated from your loop
file_name = "report_dataset_cpl-corp-mpd-prod-01082025_mlops_v01_shared.xlsx"

#Create a index to the first page

valid_tables_names_for_analysis = [
    "cat_estructuratiendas",
"coppelpayinfra_movimientos_coppelpay",
"transaccionescybersourceprestamos",
"funnel_candidatos_abonos",
"ctr_elementos_home",
"funnel_candidatosprestamos",
"his_mae_articulos",
"inflacion",
"transaccionesCybersourceAbonos",
"mae_actividad_clientescredito",
"mae_AfiliadosCoppelPay",
"mae_buildingblockpresolicitudes",
"mae_buildingblockpresolicitudes_Unificado",
"mae_clientes_digitales_con_actividad_por_region",
"mae_cat_buildingblocks",
"mae_clientesnuevos",
"mae_ClientesNuevosUnificadoFormalizacion",
"mae_conteouniversospim",
"mae_ClientesCoppelPay",
"mae_catalogoinflacion",
"mae_Convenios",
"mae_entregaentienda",
"mae_funnel_candidatos_bancoppel",
"mae_funnelsku",
"mae_funnelsku_mkp",
"mae_genesys_coppel",
"mae_inventario_rz",
"mae_inventariobodegas",
"mae_inventarioypublicacion",
"mae_nps_entregas_muebles_ropa",
"mae_npstcr",
"mae_ordenes",
"mae_dud_funnel_abandono",
"mae_eliminacion_cuenta",
"mae_diferenciaprecioscompetidores",
"mae_prestamodigital_clientes_perdidos_mensual",
"mae_tendencia_solicitudes_arg",
"mae_solicitudcredito",
"mae_seguimientocatsolicitudcredito",
"mae_fechas",
"mae_prestamos_grupo_coppel",
"mae_situacioninventario",
"mae_funnel_abandono_error",
"mae_PerfilClientePrestamos",
"mae_ventascarteras_marketplace",
"mae_ventasportienda",
"mae_prestamos_diario",
"mae_prevencion_fraudes",
"tarjetasentregadasporcanal",
"mae_Prestamos",
"mae_FunnelFormalizacionCreditoCoppelE2E",
"mae_funnel_candidatosprestamos_nuevascategorias",
"mae_contracargosar",
"mae_ventas",
"mae_ventas_diaria",
"mae_ventastienda",
"mae_contracargos",
"mae_CNO_SolicitudesCAT",
"mae_AprobacionAbonosPrueba",
"mae_AprobacionAbonos",
"mae_aprobacion",
"mae_AbonosGrupoCoppel",
"mae_abonos_aprobacion_general",
"mae_Abonos",
"Formalizacion_DUD",
"ctl_colaborador",
"mae_EmpleadosVendiendoComEnTienda",
"mae_vendedor_omnicanal_empleado",
"analytics_landing_invitado_prestamos_com",
"Abonos_Prestamos_FoliosSSFFSoftLaunch"
]

print("ava funciona")

is_a_valid_table_by_contains = lambda x: any(name.lower() in x.lower() for name in valid_tables_names_for_analysis)
is_not_a_valid_table_by_contains = lambda x: not is_a_valid_table_by_contains(x)


merge_table_sum = []
for table_name, content in info_for_page.items():
  # print(table_name)
  df_sum = content["table_sum"]
  #Replace table_name name column to Tabla
  df_sum["Tabla"] = table_name
  df_sum.rename(columns={"table_name": "Tabla"}, inplace=True)
  merge_table_sum.append(df_sum)

  if df_sum["Accesibilidad"][0] == True:
    print(" Hola")
    df_field_sum = content["table_col"]
    if "Valores faltantes" in df_field_sum.columns:
      sum_valores_faltantes = df_field_sum["Valores faltantes"].sum()
    valores_totales = df_sum["Número de filas"] * df_sum["Número de columnas"]
    df_sum["Porcentaje de valores faltantes"] = sum_valores_faltantes / valores_totales



# print(merge_table_sum)
df_merge_table_sum = pd.concat(merge_table_sum)

print("HENDRIX")
print(df_merge_table_sum)
#Filter by is_a_valid_table_by_contains
df_merge_table_sum = df_merge_table_sum[df_merge_table_sum["Tabla"].apply(is_a_valid_table_by_contains)]
print("eRICK")

df_merge_table_sum.sort_values(by=["Accesibilidad", "Número de filas", "Número de columnas", "Porcentaje de valores faltantes"], inplace=True, ascending=False)
#Add a index column
df_merge_table_sum.reset_index(drop=True, inplace=True)
df_merge_table_sum.index += 1
#Add the index as a columns called "Índice"
df_merge_table_sum.insert(0, "ID", df_merge_table_sum.index)

print("ava funciona")
#Importante columns

visual_columns = ["ID", 'Accesibilidad', 'Número de columnas', 'Número de filas',"Porcentaje de valores faltantes",
       'Fecha de creación', "Columna de periodicidad", "Periodicidad" , 'Tabla']
df_merge_table_sum = df_merge_table_sum[visual_columns]
# print(df_merge_table_sum)

#Tablas no presentes en el dataset
# df_merge_table_sum_non_tables_in_dataset = df_merge_table_sum[df_merge_table_sum["Tabla"].apply(is_not_a_valid_table_by_contains)]



"""

EXCEL CREATION

"""
with pd.ExcelWriter(file_name, engine='xlsxwriter') as writer:

    #The first page is only the dataframe df_merge_table_sum
    df_merge_table_sum.to_excel(writer, sheet_name="Resumen", index=False)

    calculated_table_names = df_merge_table_sum["Tabla"]
    exist_table_name_ny_contained_name = lambda x: any(x.lower() in name.lower() for name in calculated_table_names)

    missing_tables_names = [name for name in valid_tables_names_for_analysis if not exist_table_name_ny_contained_name(name)]
    df_missing_tables_in_dataset = pd.DataFrame({"Tabla": missing_tables_names})
    df_missing_tables_in_dataset.to_excel(writer, sheet_name="Resumen", index=False, startrow=len(df_merge_table_sum) + 2, header=["Tablas faltantes en dataset"])

    for table_name in df_merge_table_sum["Tabla"]:
    # for table_name, content in info_for_page.items():
        # Excel sheet names have a 31-character limit
        if is_a_valid_table_by_contains(table_name):
          print(table_name)
          id_page = df_merge_table_sum[df_merge_table_sum["Tabla"] == table_name].index[0]
          sheet_name = f"Tabla {id_page}"
          # sheet_name = table_name[:31]
          content = info_for_page[table_name]

          print("hola")
          # 1. Write the 'table_ref' at the top
          pd.Series([content['table_ref']]).to_excel(
              writer, sheet_name=sheet_name, index=False, header=["Table Reference"], startrow=0
          )

          # 2. Write 'table_sum' (Resumen de la tabla)
          start_row_sum = 3
          pd.Series(["Resumen de la tabla"]).to_excel(
              writer, sheet_name=sheet_name, index=False, header=False, startrow=start_row_sum
          )
          # visual_columns_field =
          # table_sum_visuable = content['table_sum'][visual_columns]
          if isinstance(content['table_sum'], pd.DataFrame):
              content['table_sum'].to_excel(writer, sheet_name=sheet_name, startrow=start_row_sum + 1)
          else:
              pd.Series([content['table_sum']]).to_excel(writer, sheet_name=sheet_name, index=False, header=False, startrow=start_row_sum + 1)

          # 3. Write 'table_col' (Descripción de las variables)
          # We calculate the next start row based on the size of the previous dataframe
          offset = len(content['table_sum']) + 6 if isinstance(content['table_sum'], pd.DataFrame) else 6
          pd.Series(["Descripción de las variables"]).to_excel(
              writer, sheet_name=sheet_name, index=False, header=False, startrow=offset
          )
          if isinstance(content['table_col'], pd.DataFrame):
              try:
                content['table_col'].to_excel(writer, sheet_name=sheet_name, startrow=offset + 1)
              except Exception as e:
                def remove_timezone(val):
                  if isinstance(val, pd.Timestamp) and val.tz is not None:
                      return val.tz_localize(None)
                  elif isinstance(val, datetime.datetime) and val.tzinfo is not None:
                      return val.replace(tzinfo=None)
                  return val

                df_foo = content['table_col'].copy()
                print(df_foo)



                df_foo['Valor mínimo'] = df_foo['Valor mínimo'].apply(remove_timezone)
                df_foo['Valor máximo'] = df_foo['Valor máximo'].apply(remove_timezone)
                print(e)
                print(table_name)
                print(content['table_col'])
                df_foo.to_excel(writer, sheet_name=sheet_name, startrow=offset + 1)
          print("adios")
          # 4. Write 'error' (Error de conexión)
          if content['error'] not in [None, " "]:
            error_offset = offset + len(content['table_col']) + 3 if isinstance(content['table_col'], pd.DataFrame) else offset + 5
            pd.Series(["Error de conexión"]).to_excel(
                writer, sheet_name=sheet_name, index=False, header=False, startrow=error_offset
            )
            print(table_name)
            print(str(content['error']))
            pd.Series([str(content['error'])]).to_excel(
                writer, sheet_name=sheet_name, index=False, header=False, startrow=error_offset + 1
            )

print(f"File '{file_name}' has been created successfully.")

ava funciona
 Hola
 Hola
 Hola
 Hola
 Hola
 Hola
 Hola
 Hola
 Hola
 Hola
 Hola
 Hola
 Hola
 Hola
 Hola
 Hola
 Hola
 Hola
 Hola
 Hola
 Hola
 Hola
 Hola
 Hola
 Hola
 Hola
 Hola
 Hola
 Hola
 Hola
 Hola
 Hola
 Hola
 Hola
 Hola
 Hola
 Hola
 Hola
 Hola
 Hola
 Hola
 Hola
 Hola
 Hola
 Hola
 Hola
 Hola
 Hola
 Hola
 Hola
 Hola
 Hola
 Hola
 Hola
 Hola
 Hola
 Hola
 Hola
 Hola
 Hola
 Hola
 Hola
 Hola
 Hola
 Hola
 Hola
 Hola
 Hola
 Hola
 Hola
 Hola
 Hola
 Hola
 Hola
 Hola
 Hola
 Hola
 Hola
 Hola
 Hola
 Hola
 Hola
 Hola
 Hola
 Hola
 Hola
 Hola
HENDRIX
    Accesibilidad  Número de columnas  Número de filas    Fecha de creación  \
0            True                   2               43  2026-04-24 11:57:48   
0            True                  10          3116901  2026-04-30 13:52:12   
0            True                  20          2105150  2026-04-30 13:54:28   
0            True                  19         60321987  2026-04-30 13:54:32   
0            True                  19          4345207  2026-0

# OLD CODE




# Operators class



In [ ]:
import pandas as pd
from typing import Tuple
def get_data_page_from_table(table: SchemaTable) -> Tuple[str, pd.DataFrame,pd.DataFrame, str]:
  # Summary of table
  stats = table.stats
  df_summary_table = pd.DataFrame(stats, index=[0])
  # Replace names Columns
  df_summary_table_readable_names_columns = {
      "num_columns" : "Número de columnas",
      "num_rows" : "Número de filas",
      "creation_time" : "Fecha de creación",
      "accessibility" : "Accesibilidad"
  }
  df_summary_table.rename(columns=df_summary_table_readable_names_columns, inplace=True)

  is_valid_table_to_calculate = stats["accessibility"]
  if not is_valid_table_to_calculate:
    error_message = table.error
    fields = table.fields
    df_field_information = pd.DataFrame({"Nombre columna": [f.name for f in fields],
                                         "Tipo": [f.eda_description.type_description.value for f in fields]})
    return table.table_id, df_summary_table, df_field_information, error_message

  else:
    try:
      query_data_columns = table.generate_query()
      df_columns_info = client.query(query_data_columns).to_dataframe()
      error_message = " "

      #df_columns_info = client.query(query_data_columns).to_dataframe()
      column_names = [f.name for f in table.fields]

      list_data_columns = []
      for field in table.fields:
        name = field.name
        type_field = field.eda_description.type_description.value
        info_column = {}
        info_column["name"] = name
        info_column["type"] = type_field
        valid_calculations_columns = filter(lambda x: x.startswith(name), df_columns_info.columns)
        for valid_column in valid_calculations_columns:
          value = df_columns_info[valid_column][0]

          if "missing_values" in valid_column:
            value = int(value)
            info_column["missing_values"] = value
          elif "count_values" in valid_column:
            value = int(value)
            info_column["count_values"] = value
          elif "min_value" in valid_column:
            # value = float(value)
            info_column["min_value"] = value
          elif "max_value" in valid_column:
            # value = float(value)
            info_column["max_value"] = value
          elif "unique_values" in valid_column:
            value = int(value)
            info_column["unique_values"] = value
          elif "std_value" in valid_column:
            value = float(value)
            info_column["std_value"] = value
          elif "mean_value" in valid_column:
            value = float(value)
            info_column["mean_value"] = value

        list_data_columns.append(info_column)

      df_columns_info = pd.DataFrame(list_data_columns)
      #Calculate percetaneje of missing values
      df_columns_info["missing_values_percentage"] = df_columns_info["missing_values"] / (df_columns_info["count_values"] + df_columns_info["missing_values"])
      #Saved with only two decimales
      df_columns_info["missing_values_percentage"] = df_columns_info["missing_values_percentage"].apply(lambda x: round(x, 2))
      # Replace names Columns
      df_columns_info_table_readable_names_columns = {
          "name" : "Nombre columna",
          "type" : "Tipo",
          "missing_values" : "Valores faltantes",
          "count_values" : "Conteo de valores",
          "min_value" : "Valor mínimo",
          "max_value" : "Valor máximo",
          "unique_values" : "Valores únicos",
          "std_value" : "Desviación estándar",
          "mean_value" : "Media",
          "missing_values_percentage" : "Porcentaje de valores faltantes"
      }
      df_columns_info.rename(columns=df_columns_info_table_readable_names_columns, inplace=True)
      #Saved with another order of columns
      #df_columns_info = df_columns_info[["Nombre columna", "Tipo", "Conteo de valores", "Valores faltantes", "Porcentaje de valores faltantes", "Valor mínimo", "Media", "Desviación estándar", "Valor máximo", "Valores únicos"]]

      return table.table_id, df_summary_table, df_columns_info, error_message
    except Exception as e:
      error_message = e.message
      return table.table_id, df_summary_table, None, error_message

info_for_page = {}
for t in tables_with_schema:
  # print(t.stats)
  # print(t.name)
  name, df_summ, df_col, error = get_data_page_from_table(t)
  name_table = t.name.split(".")[-1]
  print(name)
  info_for_page[name_table] = {
      "table_ref": name,
      "table_sum": df_summ,
      "table_col": df_col,
      "error": error,
      "table_object": t
  }
  # print(maybe_error)
  # if is_valid:
  #   df_stats = client.query(t.generate_query()).to_dataframe()
  #   print(df_stats)

#client.query(query).to_dataframe()

cpl-corp-mpd-prod-01082025.mlops_v01_shared.vw_abonos_prestamos_foliosssffsoftlaunch
cpl-corp-mpd-prod-01082025.mlops_v01_shared.vw_analytics_abono_coppel_trafico_bancoppel_app
cpl-corp-mpd-prod-01082025.mlops_v01_shared.vw_analytics_atribucion_n2_eventos_colaterales_ssff_app
cpl-corp-mpd-prod-01082025.mlops_v01_shared.vw_analytics_atribucion_n2_ssff_app
cpl-corp-mpd-prod-01082025.mlops_v01_shared.vw_analytics_atribucion_pagare_ssff_app
cpl-corp-mpd-prod-01082025.mlops_v01_shared.vw_analytics_ayuda_promotor_n2_ssff_app
cpl-corp-mpd-prod-01082025.mlops_v01_shared.vw_analytics_entradas_abonos_app
cpl-corp-mpd-prod-01082025.mlops_v01_shared.vw_analytics_funnel_abonos_app
cpl-corp-mpd-prod-01082025.mlops_v01_shared.vw_analytics_funnel_abonos_com
cpl-corp-mpd-prod-01082025.mlops_v01_shared.vw_analytics_funnel_prestamos_app
cpl-corp-mpd-prod-01082025.mlops_v01_shared.vw_analytics_funnel_prestamos_com
cpl-corp-mpd-prod-01082025.mlops_v01_shared.vw_analytics_landing_invitado_prestamos_com
cpl-

In [ ]:
# Calcular periodicidad



In [ ]:
# Retry simple metrics for especial cases
import time
def retry_metrics_for_missing_column_name(table: SchemaTable,  error_str: str, df_col: pd.DataFrame) -> pd.DataFrame:
  if "reason: accessDenied" in error_str:
      return df_col
  elif "reason: invalidQuery, location: query" in error_str:
    query_data_columns = table.generate_query()
    #Extract columns from error
    missing_column_name = ExtraQueryForFistDeniedSelect.get_first_missing_column_for_correct_query(error_str)
    #Add where clause into query_data_columns
    query_data_columns = query_data_columns + f" WHERE {missing_column_name} > (CURRENT_DATE - INTERVAL 1 YEAR);"

    print(query_data_columns)
    df_columns_info = client.query(query_data_columns).to_dataframe()
    print(df_columns_info)
    # print(df_columns_info
    error_message = " "

    #df_columns_info = client.query(query_data_columns).to_dataframe()
    column_names = [f.name for f in table.fields]

    list_data_columns = []
    for field in table.fields:
      name = field.name
      type_field = field.eda_description.type_description.value
      info_column = {}
      info_column["name"] = name
      info_column["type"] = type_field
      valid_calculations_columns = filter(lambda x: x.startswith(name), df_columns_info.columns)
      for valid_column in valid_calculations_columns:
        value = df_columns_info[valid_column][0]


        if "missing_values" in valid_column:
          value = int(value)
          info_column["missing_values"] = value
        elif "count_values" in valid_column:
          value = int(value)
          info_column["count_values"] = value
        elif "min_value" in valid_column:
          # value = float(value)
          info_column["min_value"] = value
        elif "max_value" in valid_column:
          # value = float(value)
          info_column["max_value"] = value
        elif "unique_values" in valid_column:
          value = int(value)
          info_column["unique_values"] = value
        elif "std_value" in valid_column:
          value = float(value)
          info_column["std_value"] = value
        elif "mean_value" in valid_column:
          value = float(value)
          info_column["mean_value"] = value

      list_data_columns.append(info_column)

    df_columns_info = pd.DataFrame(list_data_columns)
    #Calculate percetaneje of missing values
    df_columns_info["missing_values_percentage"] = df_columns_info["missing_values"] / (df_columns_info["count_values"] + df_columns_info["missing_values"])
    #Saved with only two decimales
    df_columns_info["missing_values_percentage"] = df_columns_info["missing_values_percentage"].apply(lambda x: round(x, 2))
    # Replace names Columns
    df_columns_info_table_readable_names_columns = {
        "name" : "Nombre columna",
        "type" : "Tipo",
        "missing_values" : "Valores faltantes",
        "count_values" : "Conteo de valores",
        "min_value" : "Valor mínimo",
        "max_value" : "Valor máximo",
        "unique_values" : "Valores únicos",
        "std_value" : "Desviación estándar",
        "mean_value" : "Media",
        "missing_values_percentage" : "Porcentaje de valores faltantes"
    }
    df_columns_info.rename(columns=df_columns_info_table_readable_names_columns, inplace=True)
    return df_columns_info
  else:
    return df_col

# print(list(filter(lambda x: x["table_col"] is None, info_for_page.values())))


special_tables = ["vw_his_mae_articulos",
                  "vw_mae_aprobacionabonosprueba",
                  "vw_his_mae_entregas",
                  "vw_mae_abonos",
                  "vw_mae_aprobacion",
                  "vw_mae_aprobacionabonos",
                  "vw_mae_diferenciaprecioscompetidores",
                  "vw_mae_ordenes",
                  "vw_mae_prestamos"]

retry_tables_for_missing_column_name = []
for t in info_for_page.keys():
  # print(info_for_page[t]["table_col"])
  if info_for_page[t]["table_col"] is None and t not in special_tables:
    retry_tables_for_missing_column_name.append(t)

# print(retry_tables_for_missing_column_name)

for t_with_error in retry_tables_for_missing_column_name:
  time_init = time.time()
  print(f"PROCESSING TABLE {t_with_error}")
  table = info_for_page[t_with_error]["table_object"]
  error_str = info_for_page[t_with_error]["error"]
  df_col = info_for_page[t]["table_col"]
  df_col_new = retry_metrics_for_missing_column_name(table, error_str, df_col)

  #Replace values into the original ones from info_for_page
  info_for_page[t]["table_col"] = df_col_new
  time_end = time.time()
  print(f"TIME TO RETRY: {time_end - time_init}")

PROCESSING TABLE vw_analytics_abono_coppel_trafico_bancoppel_app

      SELECT
        SUM(CASE WHEN fec_FechaVisita IS NULL THEN 1 ELSE 0 END) as fec_FechaVisita_missing_values,
MIN(fec_FechaVisita) as fec_FechaVisita_min_value,
MAX(fec_FechaVisita) as fec_FechaVisita_max_value,
SUM(CASE WHEN des_PaginaVisita IS NULL THEN 1 ELSE 0 END) as des_PaginaVisita_missing_values,
COUNT(des_PaginaVisita) as des_PaginaVisita_count_values,
COUNT(DISTINCT des_PaginaVisita) as des_PaginaVisita_unique_values,
SUM(CASE WHEN des_Plataforma IS NULL THEN 1 ELSE 0 END) as des_Plataforma_missing_values,
COUNT(des_Plataforma) as des_Plataforma_count_values,
COUNT(DISTINCT des_Plataforma) as des_Plataforma_unique_values,
SUM(CASE WHEN num_Visitas IS NULL THEN 1 ELSE 0 END) as num_Visitas_missing_values,
COUNT(num_Visitas) as num_Visitas_count_values,
SUM(CASE WHEN num_VisitasUnicas IS NULL THEN 1 ELSE 0 END) as num_VisitasUnicas_missing_values,
COUNT(num_VisitasUnicas) as num_VisitasUnicas_count_values,
SUM

# Creating Excel



In [ ]:
info_for_page

{'vw_abonos_prestamos_foliosssffsoftlaunch': {'table_ref': 'cpl-corp-mpd-prod-01082025.mlops_v01_shared.vw_abonos_prestamos_foliosssffsoftlaunch',
  'table_sum':    Accesibilidad  Número de columnas  Número de filas    Fecha de creación  \
  0           True                   2               43  2026-04-24 11:57:48   
  
                                        Tabla  Porcentaje de valores faltantes  
  0  vw_abonos_prestamos_foliosssffsoftlaunch                         0.372093  ,
  'table_col':       Nombre columna   Tipo  Valores faltantes  Valores únicos  \
  0     idu_FolioAbono  INDEX                  0              42   
  1  idu_FolioPrestamo  INDEX                 32              11   
  
     Conteo de valores  Porcentaje de valores faltantes  
  0                 43                             0.00  
  1                 11                             0.74  ,
  'error': ' ',
  'table_object': SchemaTable(name='vw_abonos_prestamos_foliosssffsoftlaunch', fields_count=2)},
 'vw_a

In [ ]:
!pip install xlsxwriter

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 5.9 MB/s eta 0:00:00


In [ ]:
import pandas as pd

# Assuming info_for_page is already populated from your loop
file_name = "Data_Dictionary_Report.xlsx"

#Create a index to the first page

valid_tables_names_for_analysis = [
    "cat_estructuratiendas",
"coppelpayinfra_movimientos_coppelpay",
"transaccionescybersourceprestamos",
"funnel_candidatos_abonos",
"ctr_elementos_home",
"funnel_candidatosprestamos",
"his_mae_articulos",
"inflacion",
"transaccionesCybersourceAbonos",
"mae_actividad_clientescredito",
"mae_AfiliadosCoppelPay",
"mae_buildingblockpresolicitudes",
"mae_buildingblockpresolicitudes_Unificado",
"mae_clientes_digitales_con_actividad_por_region",
"mae_cat_buildingblocks",
"mae_clientesnuevos",
"mae_ClientesNuevosUnificadoFormalizacion",
"mae_conteouniversospim",
"mae_ClientesCoppelPay",
"mae_catalogoinflacion",
"mae_Convenios",
"mae_entregaentienda",
"mae_funnel_candidatos_bancoppel",
"mae_funnelsku",
"mae_funnelsku_mkp",
"mae_genesys_coppel",
"mae_inventario_rz",
"mae_inventariobodegas",
"mae_inventarioypublicacion",
"mae_nps_entregas_muebles_ropa",
"mae_npstcr",
"mae_ordenes",
"mae_dud_funnel_abandono",
"mae_eliminacion_cuenta",
"mae_diferenciaprecioscompetidores",
"mae_prestamodigital_clientes_perdidos_mensual",
"mae_tendencia_solicitudes_arg",
"mae_solicitudcredito",
"mae_seguimientocatsolicitudcredito",
"mae_fechas",
"mae_prestamos_grupo_coppel",
"mae_situacioninventario",
"mae_funnel_abandono_error",
"mae_PerfilClientePrestamos",
"mae_ventascarteras_marketplace",
"mae_ventasportienda",
"mae_prestamos_diario",
"mae_prevencion_fraudes",
"tarjetasentregadasporcanal",
"mae_Prestamos",
"mae_FunnelFormalizacionCreditoCoppelE2E",
"mae_funnel_candidatosprestamos_nuevascategorias",
"mae_contracargosar",
"mae_ventas",
"mae_ventas_diaria",
"mae_ventastienda",
"mae_contracargos",
"mae_CNO_SolicitudesCAT",
"mae_AprobacionAbonosPrueba",
"mae_AprobacionAbonos",
"mae_aprobacion",
"mae_AbonosGrupoCoppel",
"mae_abonos_aprobacion_general",
"mae_Abonos",
"Formalizacion_DUD",
"ctl_colaborador",
"mae_EmpleadosVendiendoComEnTienda",
"mae_vendedor_omnicanal_empleado",
"analytics_landing_invitado_prestamos_com",
"Abonos_Prestamos_FoliosSSFFSoftLaunch"
]

print("ava funciona")

is_a_valid_table_by_contains = lambda x: any(name.lower() in x.lower() for name in valid_tables_names_for_analysis)
is_not_a_valid_table_by_contains = lambda x: not is_a_valid_table_by_contains(x)


merge_table_sum = []
for table_name, content in info_for_page.items():
  # print(table_name)
  df_sum = content["table_sum"]
  #Replace table_name name column to Tabla
  df_sum["Tabla"] = table_name
  df_sum.rename(columns={"table_name": "Tabla"}, inplace=True)
  merge_table_sum.append(df_sum)

  if df_sum["Accesibilidad"][0] == True:
    print(" Hola")
    df_field_sum = content["table_col"]
    sum_valores_faltantes = df_field_sum["Valores faltantes"].sum()
    valores_totales = df_sum["Número de filas"] * df_sum["Número de columnas"]
    df_sum["Porcentaje de valores faltantes"] = sum_valores_faltantes / valores_totales



# print(merge_table_sum)
df_merge_table_sum = pd.concat(merge_table_sum)

print("HENDRIX")
print(df_merge_table_sum)
#Filter by is_a_valid_table_by_contains
df_merge_table_sum = df_merge_table_sum[df_merge_table_sum["Tabla"].apply(is_a_valid_table_by_contains)]
print("eRICK")

df_merge_table_sum.sort_values(by=["Accesibilidad", "Número de filas", "Número de columnas", "Porcentaje de valores faltantes"], inplace=True, ascending=False)
#Add a index column
df_merge_table_sum.reset_index(drop=True, inplace=True)
df_merge_table_sum.index += 1
#Add the index as a columns called "Índice"
df_merge_table_sum.insert(0, "ID", df_merge_table_sum.index)

print("ava funciona")
#Importante columns
visual_columns = ["ID", 'Accesibilidad', 'Número de columnas', 'Número de filas',"Porcentaje de valores faltantes",
       'Fecha de creación', 'Tabla']
df_merge_table_sum = df_merge_table_sum[visual_columns]
# print(df_merge_table_sum)

#Tablas no presentes en el dataset
# df_merge_table_sum_non_tables_in_dataset = df_merge_table_sum[df_merge_table_sum["Tabla"].apply(is_not_a_valid_table_by_contains)]



"""

EXCEL CREATION

"""
with pd.ExcelWriter(file_name, engine='xlsxwriter') as writer:

    #The first page is only the dataframe df_merge_table_sum
    df_merge_table_sum.to_excel(writer, sheet_name="Resumen", index=False)

    calculated_table_names = df_merge_table_sum["Tabla"]
    exist_table_name_ny_contained_name = lambda x: any(x.lower() in name.lower() for name in calculated_table_names)

    missing_tables_names = [name for name in valid_tables_names_for_analysis if not exist_table_name_ny_contained_name(name)]
    df_missing_tables_in_dataset = pd.DataFrame({"Tabla": missing_tables_names})
    df_missing_tables_in_dataset.to_excel(writer, sheet_name="Resumen", index=False, startrow=len(df_merge_table_sum) + 2, header=["Tablas faltantes en dataset"])

    for table_name in df_merge_table_sum["Tabla"]:
    # for table_name, content in info_for_page.items():
        # Excel sheet names have a 31-character limit
        if is_a_valid_table_by_contains(table_name):
          print(table_name)
          id_page = df_merge_table_sum[df_merge_table_sum["Tabla"] == table_name].index[0]
          sheet_name = f"Tabla {id_page}"
          # sheet_name = table_name[:31]
          content = info_for_page[table_name]

          # 1. Write the 'table_ref' at the top
          pd.Series([content['table_ref']]).to_excel(
              writer, sheet_name=sheet_name, index=False, header=["Table Reference"], startrow=0
          )

          # 2. Write 'table_sum' (Resumen de la tabla)
          start_row_sum = 3
          pd.Series(["Resumen de la tabla"]).to_excel(
              writer, sheet_name=sheet_name, index=False, header=False, startrow=start_row_sum
          )
          # visual_columns_field =
          # table_sum_visuable = content['table_sum'][visual_columns]
          if isinstance(content['table_sum'], pd.DataFrame):
              content['table_sum'].to_excel(writer, sheet_name=sheet_name, startrow=start_row_sum + 1)
          else:
              pd.Series([content['table_sum']]).to_excel(writer, sheet_name=sheet_name, index=False, header=False, startrow=start_row_sum + 1)

          # 3. Write 'table_col' (Descripción de las variables)
          # We calculate the next start row based on the size of the previous dataframe
          offset = len(content['table_sum']) + 6 if isinstance(content['table_sum'], pd.DataFrame) else 6
          pd.Series(["Descripción de las variables"]).to_excel(
              writer, sheet_name=sheet_name, index=False, header=False, startrow=offset
          )
          if isinstance(content['table_col'], pd.DataFrame):
              content['table_col'].to_excel(writer, sheet_name=sheet_name, startrow=offset + 1)

          # 4. Write 'error' (Error de conexión)
          if content['error'] not in [None, " "]:
            error_offset = offset + len(content['table_col']) + 3 if isinstance(content['table_col'], pd.DataFrame) else offset + 5
            pd.Series(["Error de conexión"]).to_excel(
                writer, sheet_name=sheet_name, index=False, header=False, startrow=error_offset
            )
            print(table_name)
            print(str(content['error']))
            pd.Series([str(content['error'])]).to_excel(
                writer, sheet_name=sheet_name, index=False, header=False, startrow=error_offset + 1
            )

print(f"File '{file_name}' has been created successfully.")

ava funciona
 Hola
 Hola
 Hola
 Hola
 Hola
 Hola
 Hola
 Hola
 Hola
 Hola
 Hola
 Hola
 Hola
 Hola
 Hola
 Hola
 Hola
 Hola
 Hola
 Hola
 Hola
 Hola
 Hola
 Hola
 Hola
 Hola
 Hola
 Hola
 Hola
 Hola
 Hola
 Hola
 Hola
 Hola
 Hola
 Hola
 Hola
 Hola
 Hola
 Hola
 Hola
HENDRIX
    Accesibilidad Número de columnas Número de filas    Fecha de creación  \
0            True                  2              43  2026-04-24 11:57:48   
0           False                 10          280436  2026-04-30 13:52:12   
0           False                 20         1107258  2026-04-30 13:54:28   
0           False                 19        31531314  2026-04-30 13:54:32   
0            True                 19         4262282  2026-04-30 13:54:37   
..            ...                ...             ...                  ...   
0           False                 16        24539452  2026-04-24 11:58:49   
0           False               None            None                  NaN   
0            True                  7    

# Invalid query by error queries


In [ ]:
tables_with_error = list(filter(lambda t: t.error != {}, tables_with_schema))

list_error = map(lambda t: t.error["missing_query"], tables_with_error)
#
tables_with_query_error = filter(lambda t: "reason: invalidQuery, location: query" in t.error["missing_query"], tables_with_error)
tables_with_permission_error = filter(lambda t: "reason: accessDenied" in t.error["missing_query"], tables_with_error)


In [ ]:
tables_with_query_error = list(tables_with_query_error)
tables_with_permission_error = list(tables_with_permission_error)

In [ ]:
from IPython.core.logger import time
import re
from typing import Optional
import pandas as pd

a = list(map(lambda x: ExtraQueryForFistDeniedSelect.get_first_missing_column_for_correct_query(x.error["missing_query"]),tables_with_query_error))

class ExtraQueryForFistDeniedSelect:
  def get_first_missing_column_for_correct_query(str_error: str) -> Optional[str]:
    # text = "over column(s) 'fec_FechaCorte'"

    # The regex pattern
    pattern = r"over column\(s\)\s+'([^']+)'"
    # Search for the match
    match = re.search(pattern, str_error)

    if match:
        column_name = match.group(1)
        return column_name
    else:
        return None


def make_printed_queries(table):
  # print(f"Nombre de la tabla: {table.table_id}")
  # print(" ")

  metric_query = table.generate_query()
  # print("=")
  missing_columns_for_query = ExtraQueryForFistDeniedSelect.get_first_missing_column_for_correct_query(table.error["missing_query"])
  select_All_query = "SELECT * FROM " + table.table_id
  where_text = f" WHERE {missing_columns_for_query} > (CURRENT_DATE - INTERVAL 1 year);"
  query_final = select_All_query + where_text
  counting_query = f"""
SELECT
  COUNT( DISTINCT {missing_columns_for_query})
FROM
  `{table.table_id}`
WHERE
  {missing_columns_for_query} > (CURRENT_DATE - INTERVAL 1 YEAR);"""

  print(f"Processing table {table.table_id}")
  t_init = time.time()
  result = client.query(counting_query).to_dataframe()
  print(result)
  # client.query(query_final).to_dataframe()
  delta = time.time() - t_init
  print(f"Query time: {delta} seconds")
  # return {"table_name": table.table_id, "query": query_final}

# df = pd.DataFrame(list(map(make_printed_queries, tables_with_query_error)))
# df.to_csv("index_for_temporal_variables.csv")

list(map(make_printed_queries, tables_with_query_error))

Processing table cpl-corp-mpd-prod-01082025.mlops_v01_shared.vw_analytics_abono_coppel_trafico_bancoppel_app
   f0_
0   28
Query time: 2.679785966873169 seconds
Processing table cpl-corp-mpd-prod-01082025.mlops_v01_shared.vw_analytics_atribucion_n2_eventos_colaterales_ssff_app
   f0_
0  365
Query time: 4.477593421936035 seconds
Processing table cpl-corp-mpd-prod-01082025.mlops_v01_shared.vw_analytics_atribucion_n2_ssff_app
   f0_
0  365
Query time: 3.2933971881866455 seconds
Processing table cpl-corp-mpd-prod-01082025.mlops_v01_shared.vw_analytics_ayuda_promotor_n2_ssff_app
   f0_
0   14
Query time: 3.187408685684204 seconds
Processing table cpl-corp-mpd-prod-01082025.mlops_v01_shared.vw_analytics_funnel_abonos_app
   f0_
0  364
Query time: 3.781503915786743 seconds
Processing table cpl-corp-mpd-prod-01082025.mlops_v01_shared.vw_analytics_funnel_abonos_com
   f0_
0  357
Query time: 3.504108428955078 seconds
Processing table cpl-corp-mpd-prod-01082025.mlops_v01_shared.vw_ctr_elementos_h

[None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None]